In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.shape

(891, 12)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [12]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

# Data Quality Report

## Initial Dataset Inspection

The Titanic dataset was loaded successfully.

Data quality checks performed:
- Dataset shape
- Missing values
- Duplicate rows
- Data types

In [10]:
rows_before = df.shape[0]
nulls_before = df.isnull().sum().sum()
duplicates_before = df.duplicated().sum()

print("Rows:", rows_before)
print("Null Values:", nulls_before)
print("Duplicates:", duplicates_before)

Rows: 891
Null Values: 866
Duplicates: 0


In [15]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [16]:
df['Age'] = df['Age'].fillna(df['Age'].median())

## Missing Value Handling

Age column had missing values.

Median imputation was used because Age is a numeric column and median is less affected by extreme values than mean.

In [17]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

Embarked is a categorical column.

Mode imputation was used because the most frequently occurring category is an appropriate replacement.

In [18]:
df.drop('Cabin', axis=1, inplace=True)

Cabin contained a very large number of missing values.

The column was removed because imputing such a large amount of missing data would reduce reliability.

In [19]:
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [20]:
df.duplicated().sum()

np.int64(0)

## Duplicate Handling

Duplicate rows were checked using pandas duplicated().

Any duplicate rows found were removed using drop_duplicates().

In [21]:
df['Sex'].unique()

<StringArray>
['male', 'female']
Length: 2, dtype: str

In [22]:
df['Embarked'].unique()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

In [23]:
df['Sex'] = df['Sex'].str.capitalize()

In [24]:
df['Sex'].unique()

<StringArray>
['Male', 'Female']
Length: 2, dtype: str

In [26]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
dtype: object

In [27]:
df['PassengerId'] = df['PassengerId'].astype(str)

In [28]:
df.dtypes

PassengerId        str
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
dtype: object

## Data Type Correction

PassengerId was converted from integer to string.

This column represents a unique identifier and is not intended for numerical calculations.

In [29]:
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)

IQR = Q3 - Q1

print("Q1 =", Q1)
print("Q3 =", Q3)
print("IQR =", IQR)

Q1 = 22.0
Q3 = 35.0
IQR = 13.0


In [30]:
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

Lower Limit: 2.5
Upper Limit: 54.5


In [31]:
outliers = df[(df['Age'] < lower_limit) | (df['Age'] > upper_limit)]

print("Number of Outliers:", len(outliers))

Number of Outliers: 66


In [32]:
outliers[['PassengerId', 'Age']]

,PassengerId,Age
7,8,2.00
11,12,58.00
15,16,55.00
16,17,2.00
33,34,66.00
...,...,...
827,828,1.00
829,830,62.00
831,832,0.83
851,852,74.00


## Outlier Detection

Outliers in the Age column were identified using the IQR method.

The detected outliers were retained because they may represent valid passenger ages rather than data entry errors.

No outliers were removed.

In [33]:
rows_after = df.shape[0]
nulls_after = df.isnull().sum().sum()
duplicates_after = df.duplicated().sum()

In [34]:
summary = pd.DataFrame({
    'Metric': ['Row Count', 'Null Values', 'Duplicate Rows'],
    'Before Cleaning': [rows_before, nulls_before, duplicates_before],
    'After Cleaning': [rows_after, nulls_after, duplicates_after]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,891,891
1,Null Values,866,0
2,Duplicate Rows,0,0


# Before vs After Cleaning Summary

The dataset quality improved significantly after cleaning.

- Missing values were handled.
- Duplicate rows were checked.
- Data types were corrected.
- Formatting was standardized.
- Outliers were identified and documented.

The cleaned dataset is now ready for analysis.

In [35]:
df.to_csv("Titanic_Cleaned.csv", index=False)

In [36]:
import os

os.listdir()

['.ipynb_checkpoints',
 'Task2_DataCleaning.ipynb',
 'Titanic_Cleaned.csv',
 'train.csv']

# Conclusion

The Titanic dataset was successfully cleaned and prepared for analysis.

Cleaning steps included:

1. Data quality assessment
2. Missing value treatment
3. Duplicate checking
4. Data standardization
5. Data type correction
6. Outlier detection using the IQR method
7. Creation of a before-versus-after quality summary

The final cleaned dataset was exported as Titanic_Cleaned.csv.